# 04 - Logistic Regression
Load hai model đã train: baseline không xử lý imbalance và class weight. Evaluation đầy đủ được thực hiện ở Phase 4.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("04-logistic-regression")
    .config("spark.cores.max", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/26 03:08:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.9
Spark master: spark://spark-master:7077


In [3]:
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel
test = spark.read.parquet('hdfs:///financial/processed/ml_splits/test')
preprocessor = PipelineModel.load('hdfs:///financial/models/preprocessor')
features = preprocessor.transform(test.limit(1000))
lr_none = LogisticRegressionModel.load('hdfs:///financial/models/logistic_regression/none')
lr_weighted = LogisticRegressionModel.load('hdfs:///financial/models/logistic_regression/class_weight')

In [4]:
lr_weighted.transform(features).select('transaction_id', 'class', 'probability', 'prediction').show(20, truncate=False)

[Stage 33:>                                                         (0 + 1) / 1]

+--------------+-----+------------------------------------------+----------+
|transaction_id|class|probability                               |prediction|
+--------------+-----+------------------------------------------+----------+
|18881799      |0    |[0.8755051186987475,0.12449488130125252]  |0.0       |
|18881817      |0    |[0.9755900644789065,0.024409935521093495] |0.0       |
|18881925      |0    |[0.9023733882629815,0.09762661173701848]  |0.0       |
|18882086      |0    |[0.9926551402369329,0.007344859763067091] |0.0       |
|18882335      |0    |[0.9958783581513486,0.004121641848651381] |0.0       |
|18882402      |0    |[0.4189118386078511,0.5810881613921489]   |1.0       |
|18882468      |0    |[0.8184077752349044,0.18159222476509562]  |0.0       |
|18882470      |0    |[0.0011854201535120496,0.9988145798464879]|1.0       |
|18882501      |0    |[0.9718979513188414,0.02810204868115862]  |0.0       |
|18882560      |0    |[0.9995827286271038,4.1727137289615523E-4]|0.0       |